## Caching de Funciones
El caché es un término muy usado en informática, y hace referencia al almacenamiento de resultados previos para su posterior reutilización, lo que permite reducir el tiempo de respuesta. Por ejemplo, si llamamos a una función con un determinado parámetro y acto seguido realizamos la misma llamada, sería interesante reutilizar el primer resultado para no tener que calcularlo otra vez. Existen por lo tanto dos posibilidades:

* Si ejecutamos la función y el resultado no ha sido calculado con anterioridad, se calcula y se almacena por si fuera útil en el futuro. Esto se conoce como `cache miss`.
* Si ejecutamos la función y el caché tiene almacenado el resultado para esa operación, en vez de calcular otra vez la salida la podemos reutilizar, lo que se conoce como `cache hit`. Dado que estamos reutilizando un valor ya calculado, generalmente el tiempo de respuesta será menor.

Por suerte, Python nos permite añadir `caching` a nuestras funciones, pero antes de implementarlo es conveniente hacer un análisis sobre nuestro programa y determinar si merece la pena. Algunas cosas a tener en cuenta:

* El caching es especialmente útil cuando trabajamos con funciones muy intensivas en cálculo, lo que hace que reutilizar el valor del caché reduzca notablemente el tiempo de respuesta.
* Es necesario conocer (a nivel estadístico) la distribución de los argumentos con los que se llama la función. Si la función bajo estudio se llama con valores muy dispares y apenas repetidos, el caching poco ayudará, ya que apenas tendremos un `cache hit`.
* El uso de un caché puede mejorar el tiempo de respuesta, pero frecuentemente se paga en un incremento del uso de memoria. También es necesario decidir el número de valores a almacenar.

A continuación veremos como implementar caching en Python, pudiendo hacerlo con diccionarios o utilizando la librería `functools`. Para ejemplificarlo, veremos como implementar un caché en nuestro código de números primos visto anteriormente, empleando ambas formas.

In [ ]:
def es_primo(num:int) -> bool:
    """ Función que valida si un número es primo.

    Args:
        num (int): Número a validar.

    Returns:
        bool: True si el número es primo, False en caso contrario.
    """
    for n in range(2, num):
        if num % n == 0:
            return False
    return True

## Caching con Diccionarios
La primera forma de realizarlo es usando un diccionario como caché. Nótese que este es un ejemplo didáctico, y que obvia algunos factores. Como puedes ver tenemos claramente diferenciado el cache hit y el cache miss. Si el valor no está en el caché se calcula y se devuelve.

In [ ]:
def es_primo_con_cache(num:int, _cache:dict[int, bool]={}) -> bool:
    """ Función que valida si un número es primo.

    Args:
        num (int): Número a validar.
        cache (dict[int, bool]): Diccionario que almacena los valores calculados.
    Returns:
        bool: True si el número es primo, False en caso contrario.
    """
    if num not in _cache:
        _cache[num] = True
        for n in range(2, num):
            if num % n == 0:
                _cache[num] = False
                break
    return _cache[num]

Dado que la función `es_primo` es bastante intensivo en cálculo, cuando usamos números grandes el ahorro puede ser muy significativo.

En el siguiente código podemos ver como la primera vez que ejecutamos la función, se tardan `segundos`, ya que el resultado tiene que ser calculado. Sin embargo la segunda vez que la llamamos con la misma entrada, tenemos un `cache hit`, por lo que el valor ya no es calculado sino recuperado del caché, tardando `microsegundos`.

In [ ]:
import time
tic = time.time()
es_primo(25565479)
print('El tiempo que demoró en calcularlo con la función normal es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache(25565479)
print('El tiempo que demoró en calcularlo sin utilizar la cache es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache(25565479)
print('El tiempo que demoró en calcularlo utilizando la cache es: ', time.time() - tic)

El tiempo que demoró en calcularlo con la función normal es:  4.586920976638794
El tiempo que demoró en calcularlo sin utilizar la cache es:  1.8825640678405762
El tiempo que demoró en calcularlo utilizando la cache es:  7.772445678710938e-05


## Caching con functools y lru_cache

La segunda forma de realizarlo, y un poco más sofisticada y es usando `lru_cache`, un decorador que viene con la librería estándar `functools`. La mayor ventaja es que no necesitamos modificar la función. Nótese que `maxsize` nos permite indicar el número máximo de valores que queremos almacenar en el caché.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=32)
def es_primo_con_cache_lru(num:int) -> bool:
    """ Función que valida si un número es primo.

    Args:
        num (int): Número a validar.

    Returns:
        bool: True si el número es primo, False en caso contrario.
    """
    for n in range(2, num):
        if num % n == 0:
            return False
    return True

Por lo tanto si ahora llamamos a nuestra función con los mismos valores, podemos ver como la primera vez tarda 3.9 segundos, pero la segunda apenas tarda unos microsegundos.

In [ ]:
import time
tic = time.time()
es_primo(25565479)
print('El tiempo que demoró en calcularlo con la función normal es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache(25565479, {})
print('El tiempo que demoró en calcularlo sin utilizar la cache es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache(25565479)
print('El tiempo que demoró en calcularlo utilizando la cache es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache_lru(25565479)
print('El tiempo que demoró en calcularlo utilizando sin utilizar cache lru es: ', time.time() - tic)

tic = time.time()
es_primo_con_cache_lru(25565479)
print('El tiempo que demoró en calcularlo utilizando sin utilizar cache lru es: ', time.time() - tic)


El tiempo que demoró en calcularlo con la función normal es:  1.902358055114746
El tiempo que demoró en calcularlo sin utilizar la cache es:  1.9353320598602295
El tiempo que demoró en calcularlo utilizando la cache es:  7.772445678710938e-05
El tiempo que demoró en calcularlo utilizando sin utilizar cache lru es:  1.8817930221557617
El tiempo que demoró en calcularlo utilizando sin utilizar cache lru es:  7.677078247070312e-05


En el caso de que queramos limpiar el caché de nuestra función, podemos invocar a la función `cache_clear`.



In [ ]:
es_primo_con_cache_lru.cache_clear()